# long-route-planner: fly the 30 km route with the 027.12 filter

# Imports

In [ ]:
import gc
import json
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.data.google_maps import create_session, load_api_key, satellite_tile_template
from uavloc.data.satellite import GeoImage
from uavloc.data.tiles import ESRI_WORLD_IMAGERY, XyzTileSource, geo_mosaic
from uavloc.geo import GeoBounds, LatLon
from uavloc.matching.registration import apply_shift, phase_shift
from uavloc.filtering.belief import WindowConfig
from uavloc.filtering.particles import ParticleFilter, ParticleFilterConfig
from uavloc.simulation.arms import HeadSpec, build_correlator
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.ins import InsSimulator
from uavloc.simulation.route import Pose, sample_route
from uavloc.simulation.schedule import EveryInterval, FixSchedule, WhereLocalizable
from uavloc.simulation.sequence import SKIPPED_DECISIONS, FlightRun, load_flight_set
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
ROUTES = ["best", "shortest"]   # from plan.ipynb
ARMS = ["every-5s", "planner"]  # every-5s: today's clock; planner: fix where the blind map says localizable
SEEDS = [0, 1, 2]               # INS realisation and particle seed
SEED = 42
DEVICE = "cpu"
ROUTE_SET = "sim-flights"
PLAN_DIR = "pocs/long-route-planner/runs/plan"
OUT_DIR = "pocs/long-route-planner/runs/fly"
FIGURE_DIR = "pocs/long-route-planner/figures"
TILE_CACHE = "data/derived/tile-cache"
ZOOM = 18
GOOGLE_KEY = "google-satellite"   # the drone sees Google imagery; the onboard map is the planner's Esri
CHUNK_M = 2000.0                  # route length per map raster
CHUNK_MARGIN_M = 300.0            # room for the search window around a drifted estimate
MIN_SHIFT_PEAK = 15.0             # ua-village-multidate's co-registration acceptance
MAX_SHIFT_PX = 30.0
CORRECTION_INTERVAL_S = 5.0       # every-5s arm (027.12)
BLIND_THRESHOLD = 0.5             # planner arm: the planner's own "blind" cut (routes.BLIND_THRESHOLD)
MIN_FIX_SPACING_M = 50.0          # planner arm: 5 s at 10 m/s, so fix density on good ground matches every-5s
INITIAL_ERROR_M = 0.0
SIGMA0_M = 1.0
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
HEAD_PATH = "experiments/023-learned-correlation-head/artifacts/head-023.3.pt"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
MATCH_GSD_M = 0.385
CLASS_WEIGHTS = [0.0, 1.0, 1.0, 0.5, 0.0, 0.5]
N_PARTICLES = 2000
TEMPERATURE = 0.5
Q_POS_M_PER_SQRT_S = 2.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
RECOVERED_M = 10.0
MAX_POSES = None                  # first N poses of each route, for dry runs

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, _ = load_flight_set(ROUTE_SET)
ins_config = replace(set_config.ins, initial_error_m=INITIAL_ERROR_M)
plan_dir, out_dir, figure_dir = REPO_ROOT / PLAN_DIR, REPO_ROOT / OUT_DIR, REPO_ROOT / FIGURE_DIR
out_dir.mkdir(parents=True, exist_ok=True)
planned = json.loads((plan_dir / "routes.json").read_text())

In [ ]:
def load_blind_maps(path) -> list[GeoImage]:
    data = np.load(path)
    legs = sorted((k for k in data.files if not k.endswith("_bounds")), key=lambda k: int(k[3:]))
    return [GeoImage(data[k], LatLon(*data[f"{k}_bounds"][:2]), LatLon(*data[f"{k}_bounds"][2:])) for k in legs]

In [ ]:
blind_maps = load_blind_maps(plan_dir / "blind_maps.npz")
poses = {name: sample_route([LatLon(*p) for p in planned["routes"][name]], set_config.speed_mps, set_config.step_s,
                            set_config.altitude_m)[:MAX_POSES] for name in ROUTES}
{name: (len(p), p[-1].t_s * set_config.speed_mps / 1e3) for name, p in poses.items()}

In [ ]:
def chunk_slices(route_poses: list[Pose], chunk_m: float) -> list[slice]:
    poses_per_chunk = int(chunk_m / (set_config.speed_mps * set_config.step_s))
    return [slice(i, i + poses_per_chunk) for i in range(0, len(route_poses), poses_per_chunk)]

In [ ]:
api_key = load_api_key()
google = XyzTileSource(GOOGLE_KEY, satellite_tile_template(api_key, create_session(api_key)))

In [ ]:
def chunk_rasters(chunk: list[Pose]) -> tuple[GeoImage, GeoImage, dict]:
    """Esri map and Google drone raster over one chunk, Google co-registered onto Esri when the peak is reliable."""
    bounds = GeoBounds.around([p.position for p in chunk], CHUNK_MARGIN_M)
    esri = geo_mosaic(ESRI_WORLD_IMAGERY, bounds, ZOOM, REPO_ROOT / TILE_CACHE)
    drone = geo_mosaic(google, bounds, ZOOM, REPO_ROOT / TILE_CACHE)
    dx, dy, ratio = phase_shift(esri.pixels.mean(2), drone.pixels.astype(np.float32).mean(2))
    applied = ratio >= MIN_SHIFT_PEAK and np.hypot(dx, dy) <= MAX_SHIFT_PX
    if applied:
        drone = drone.with_pixels(apply_shift(drone.pixels, dx, dy))
    return esri, drone, {"shift_m": float(np.hypot(dx, dy) * esri.mpp), "peak_ratio": ratio, "applied": bool(applied)}

# Model

In [ ]:
CORRELATOR = {"method": "segmentation", "seg_architecture": "lraspp", "seg_checkpoint_drone": SEG_CHECKPOINT,
              "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_channels": "all", "drone_seg_gsd_m": MATCH_GSD_M,
              "match_gsd_m": MATCH_GSD_M, "matching": "weighted-ncc", "class_weights": CLASS_WEIGHTS, "window": "rect",
              "search_scale": 1.7, "fov_w_deg": set_config.fov_w_deg, "sat_feature_crop_pct": 0.03,
              "drone_feature_crop_pct": 0.0}
HEAD = HeadSpec(REPO_ROOT / HEAD_PATH, tuple(HEAD_ENCODER_CHANNELS), HEAD_REFINE_CHANNELS)
FILTER_CONFIG = ParticleFilterConfig(N_PARTICLES, TEMPERATURE, Q_POS_M_PER_SQRT_S,
                                     window=WindowConfig(WINDOW_CAP_HALF_M, WINDOW_MARGIN_M))

In [ ]:
def make_schedule(arm: str) -> FixSchedule:
    if arm == "every-5s":
        return EveryInterval(CORRECTION_INTERVAL_S)
    return WhereLocalizable(blind_maps, BLIND_THRESHOLD, MIN_FIX_SPACING_M)

In [ ]:
def start_runs(route_poses: list[Pose]) -> dict[tuple[str, int], FlightRun]:
    """Every arm x seed; a seed's INS realisation is shared by both arms. The map is set per chunk."""
    return {(arm, seed): FlightRun(route_poses[0], InsSimulator(ins_config, seed), ParticleFilter(None, FILTER_CONFIG, seed),
                                   make_schedule(arm), SIGMA0_M)
            for arm in ARMS for seed in SEEDS}

In [ ]:
def fly_route(route_poses: list[Pose]) -> tuple[dict[tuple[str, int], FlightRun], list[dict]]:
    """All runs advance together chunk by chunk, so only one chunk's map is in memory."""
    runs, chunks = start_runs(route_poses), []
    for part in tqdm(chunk_slices(route_poses, CHUNK_M), desc="chunks"):
        esri, drone, shift = chunk_rasters(route_poses[part])
        correlator, camera = build_correlator(esri, CORRELATOR, DEVICE, HEAD), RasterCamera(drone, CameraConfig(set_config.fov_w_deg))
        for run in runs.values():
            run.estimator.correlator = correlator
            for pose in route_poses[part]:
                run.step(pose, camera)
        chunks.append({"first_pose": part.start, **shift})
        del esri, drone, correlator, camera
        gc.collect()
    return runs, chunks

# Run

In [ ]:
flown, chunk_log = {}, {}
for name in ROUTES:
    flown[name], chunk_log[name] = fly_route(poses[name])

In [ ]:
def summary(track: pd.DataFrame, fixes: pd.DataFrame) -> dict:
    updates = fixes["decision"].eq("update") if len(fixes) else pd.Series(dtype=bool)
    return {"end_error_m": track["est_error_m"].iloc[-1], "mean_error_m": track["est_error_m"].mean(),
            "median_error_m": track["est_error_m"].median(), "p90_error_m": track["est_error_m"].quantile(0.9),
            "max_error_m": track["est_error_m"].max(), "end_error_ins_m": track["ins_error_m"].iloc[-1],
            "fixes": len(fixes), "updates": int(updates.sum()),
            "skipped_share": float(fixes["decision"].isin(SKIPPED_DECISIONS).mean()) if len(fixes) else np.nan,
            "time_within_10m": float((track["est_error_m"] <= RECOVERED_M).mean())}

In [ ]:
tables = {(name, arm, seed): run.tables() for name, runs in flown.items() for (arm, seed), run in runs.items()}
per_flight = pd.DataFrame([{"route": n, "arm": a, "seed": s, **summary(*t)} for (n, a, s), t in tables.items()])
per_flight.round(2)

In [ ]:
by_arm = per_flight.groupby(["route", "arm"]).mean(numeric_only=True).drop(columns="seed")
by_arm.round(2)

In [ ]:
pd.DataFrame([{"route": n, **c} for n, cs in chunk_log.items() for c in cs]).round(2)

In [ ]:
per_flight.to_csv(out_dir / "flights.csv", index=False)
for (n, a, s), (track, fixes) in tables.items():
    track.to_csv(out_dir / f"track-{n}-{a}-{s}.csv", index=False)
    fixes.to_csv(out_dir / f"fixes-{n}-{a}-{s}.csv", index=False)
(out_dir / "results.json").write_text(json.dumps({"by_arm": by_arm.reset_index().to_dict("records"),
                                                  "chunks": chunk_log}, indent=1, default=float))

# Plots

In [ ]:
def blind_along(route_poses: list[Pose]) -> np.ndarray:
    lookup = WhereLocalizable(blind_maps, BLIND_THRESHOLD, MIN_FIX_SPACING_M)
    return np.array([lookup.blind_share(p.position) for p in route_poses])

In [ ]:
ARM_COLOURS = {"every-5s": "tab:red", "planner": "tab:green"}
fig, axes = plt.subplots(len(ROUTES), 1, figsize=(16, 4.2 * len(ROUTES)), sharey=True, squeeze=False)
for ax, name in zip(axes[:, 0], ROUTES):
    km = np.array([p.t_s for p in poses[name]]) * set_config.speed_mps / 1e3
    blind = blind_along(poses[name]) >= BLIND_THRESHOLD
    ax.fill_between(km, 0, 1, where=blind, transform=ax.get_xaxis_transform(), color="0.85", lw=0, label="planner: blind")
    for (n, arm, seed), (track, _) in tables.items():
        if n == name:
            ax.plot(km, track["est_error_m"], color=ARM_COLOURS[arm], lw=0.9, alpha=0.8,
                    label=arm if seed == SEEDS[0] else None)
    ax.plot(km, tables[(name, ARMS[0], SEEDS[0])][0]["ins_error_m"], color="0.4", ls="--", lw=0.8, label="INS only (seed 0)")
    ax.set_yscale("log")
    ax.set_ylim(0.5, 2000)
    ax.set_xlabel("distance along route, km")
    ax.set_ylabel("position error, m")
    ax.set_title(f"{name} route: error along the flight, {len(SEEDS)} INS seeds per arm")
    ax.legend(loc="upper left", ncol=4)
fig.tight_layout()
fig.savefig(figure_dir / "error-along-route.png", dpi=130)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4.5))
for name, colour in zip(ROUTES, ("black", "tab:blue")):
    p = poses[name]
    ax.plot([q.position.lon for q in p], [q.position.lat for q in p], color=colour, lw=1, label=f"{name} (truth)")
for (n, arm, seed), (track, fixes) in tables.items():
    if n == ROUTES[0] and seed == SEEDS[0] and len(fixes):
        rows = track.iloc[fixes["step"]]
        ax.scatter(rows["true_lon"], rows["true_lat"], s=6, color=ARM_COLOURS[arm], label=f"{arm} fixes ({len(fixes)})",
                   zorder=4)
ax.set_aspect(1 / np.cos(np.radians(poses[ROUTES[0]][0].position.lat)))
ax.legend(loc="lower right")
ax.set_title(f"Where each arm attempts fixes, {ROUTES[0]} route, seed {SEEDS[0]}")
fig.tight_layout()
fig.savefig(figure_dir / "fix-locations.png", dpi=130)